In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker

import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
from cartopy.feature import ShapelyFeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

try:
    from pyproj import Geod
except ImportError:
    Geod = None

In [ ]:
# =============================================================================
# 1. USER SETTINGS
# =============================================================================

# The uploaded file name may start with a Cyrillic "с" in "сoordinates".
# The code checks both Cyrillic and Latin variants automatically.
CSV_FILE_CANDIDATES = [
    # Prefer the corrected file if it is present.
    Path("coordinates_for_map_corrected.csv"),
    Path("/mnt/data/coordinates_for_map_corrected.csv"),

    # Original file names. The first one starts with a Cyrillic "с" (U+0441).
    Path("сoordinates_for_map.csv"),
    Path("coordinates_for_map.csv"),
    Path("/mnt/data/сoordinates_for_map.csv"),
    Path("/mnt/data/coordinates_for_map.csv"),
]

# Map extent: [west_lon, east_lon, south_lat, north_lat], degrees.
# By default, the script computes a tighter extent from the sampling points.
# The crop is intentionally asymmetric in latitude: the southern margin is
# smaller to avoid showing too much land, while the northern margin leaves more
# open sea above the points.
AUTO_COMPUTE_MAP_EXTENT_FROM_POINTS = True

# Used only if AUTO_COMPUTE_MAP_EXTENT_FROM_POINTS = False.
# This manual crop is already much tighter than the original [10, 170, 48, 86].
MAP_EXTENT = [60, 100, 76, 84.5] #[50, 125, 75, 85] #[25, 165, 62, 86]

# Margins used for the automatic crop. Increase POINT_MARGIN_SOUTH_LAT_DEG if
# you want to show more coastline/land; decrease it if too much land is visible.
POINT_MARGIN_LON_DEG = 5.0
POINT_MARGIN_SOUTH_LAT_DEG = 1.6
POINT_MARGIN_NORTH_LAT_DEG = 2.2

# Hard limits for the automatic crop, useful for Arctic maps.
AUTO_EXTENT_MIN_SOUTH_LAT = 60.0
AUTO_EXTENT_MAX_NORTH_LAT = 86.5
AUTO_EXTENT_MIN_WEST_LON = 0.0
AUTO_EXTENT_MAX_EAST_LON = 180.0

# If True, points outside MAP_EXTENT will not be plotted.
# If False, all valid points are passed to Cartopy and are clipped by the axes.
FILTER_POINTS_TO_EXTENT = True

# Output folder and output quality.
OUTPUT_DIR = Path("map_outputs")
OUTPUT_DPI = 600
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SHOW_FIGURES_IN_NOTEBOOK = True

# Natural Earth vector data.
# "50m" is a good compromise: detailed enough and not too heavy.
# Use "10m" for maximum coastline detail, but it is slower and creates larger PDF/SVG.
NATURAL_EARTH_RESOLUTION = "50m"  # allowed: "10m", "50m", "110m"

# If the required land layer is missing, Cartopy will try to download it.
# If your environment has no internet, install cartopy_offlinedata as described above.
ALLOW_NATURAL_EARTH_DOWNLOAD = True

# Optional vector layers. Land is always required. These optional layers improve
# the map but are skipped with a warning if unavailable.
DRAW_COASTLINE = True
DRAW_LAKES = True
DRAW_COUNTRY_BORDERS = True
DRAW_RIVERS = False  # disabled to avoid the download problem seen in log_2.txt

# Projection center for the polar maps.
POLAR_CENTRAL_LONGITUDE = 90
POLAR_TRUE_SCALE_LATITUDE = 75

# Graticule settings.
MERIDIANS = np.arange(20, 181, 20)
PARALLELS = np.arange(50, 87, 10)

# Sampling point appearance.
POINT_SIZE = 70
POINT_FACE_COLOR = "#e41a1c"
POINT_EDGE_COLOR = "#7f0000"

# Drawing order. Larger zorder means the element is drawn later/on top.
# Reviewer-readable labels and graticule labels are intentionally above points.
LAND_ZORDER = 5
LAKES_ZORDER = 7
BORDERS_ZORDER = 8
RIVERS_ZORDER = 9
COASTLINE_ZORDER = 10
GRID_ZORDER = 35
POINT_ZORDER = 100
MAP_LABEL_ZORDER = 140
GRATICULE_LABEL_ZORDER = 150
ANNOTATION_ZORDER = 160

# Scale bar.
SCALEBAR_LENGTH_KM = 500

# The scale bar is drawn in a map corner. It is screen-horizontal, but its
# length is computed from the real geodesic distance between its endpoints.
# Allowed values: "lower right", "lower left", "upper right", "upper left".
SCALEBAR_CORNER = "lower right"
SCALEBAR_MARGIN_X_AXES = 0.055
SCALEBAR_MARGIN_Y_AXES = 0.075
SCALEBAR_TICK_HEIGHT_AXES = 0.018
SCALEBAR_LINEWIDTH = 4.0

# Basemap source/license note requested by reviewer.
BASEMAP_SOURCE_TEXT = (
    f"Basemap: Natural Earth vector data ({NATURAL_EARTH_RESOLUTION}), "
    "public domain; rendered with Cartopy"
)

# Pure vector map colors. These are intentionally simple and publication-friendly.
OCEAN_COLOR = "#c9ddec"
LAND_COLOR = "#eeeeee"
LAND_EDGE_COLOR = "#555555"
LAKE_COLOR = OCEAN_COLOR
BORDER_COLOR = "#777777"
GRID_COLOR = "#444444"

DATA_PROJECTION = ccrs.PlateCarree()


# =============================================================================
# 2. FILE AND COORDINATE HELPERS
# =============================================================================


def find_coordinate_file(candidates):
    """Return the first existing CSV file from a list of candidates."""
    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Fallback: search for corrected or original coordinate CSVs.
    for folder in [Path("."), Path("/mnt/data")]:
        if folder.exists():
            for pattern in ["*coordinates_for_map_corrected.csv", "*ordinates_for_map.csv"]:
                for candidate in folder.glob(pattern):
                    return candidate

    raise FileNotFoundError(
        "Coordinate CSV was not found. Put сoordinates_for_map.csv / "
        "coordinates_for_map.csv in the working folder or update CSV_FILE_CANDIDATES."
    )


def parse_decimal_series(series):
    """Convert decimal-comma or decimal-point coordinate strings to numeric values."""
    return pd.to_numeric(
        series.astype(str)
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def infer_coordinate_columns(df):
    """Infer latitude and longitude columns from typical names."""
    cols = list(df.columns)
    normalized = {col: re.sub(r"[^a-zа-я0-9]+", "", col.lower()) for col in cols}

    lat_col = None
    lon_col = None

    # Prefer the exact names from the uploaded CSV.
    for col in cols:
        if col.strip().lower() == "coordinates, n":
            lat_col = col
        if col.strip().lower() == "coordinates, e":
            lon_col = col

    if lat_col is None:
        for col, norm in normalized.items():
            if any(key in norm for key in ["latitude", "lat", "coordinatesn", "coordn"]):
                lat_col = col
                break

    if lon_col is None:
        for col, norm in normalized.items():
            if any(key in norm for key in ["longitude", "long", "lng", "lon", "coordinatese", "coorde"]):
                lon_col = col
                break

    # Last-resort fallback for a two-column CSV.
    if (lat_col is None or lon_col is None) and len(cols) >= 2:
        # Try to identify numeric columns by value ranges.
        numeric_candidates = []
        for col in cols:
            values = parse_decimal_series(df[col])
            valid = values.dropna()
            if len(valid) > 0:
                numeric_candidates.append((col, valid.min(), valid.max()))
        for col, vmin, vmax in numeric_candidates:
            if lat_col is None and -90 <= vmin <= 90 and -90 <= vmax <= 90:
                lat_col = col
            elif lon_col is None and -180 <= vmin <= 180 and -180 <= vmax <= 180:
                lon_col = col

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not infer latitude/longitude columns. "
            f"Available columns: {cols}. Rename them to 'Coordinates, N' and 'Coordinates, E'."
        )

    return lat_col, lon_col


def read_coordinate_csv(path):
    """
    Read the coordinate CSV robustly.

    The coordinate values may contain decimal commas and be quoted, e.g.
    "72,0349". In that case pandas auto-detection (sep=None) can fail, so
    comma-separated CSV is tried explicitly first.
    """
    attempts = [
        dict(sep=",", engine="python"),
        dict(sep=";", engine="python"),
        dict(sep="\t", engine="python"),
        dict(sep=None, engine="python"),
    ]

    last_error = None
    for kwargs in attempts:
        try:
            df_try = pd.read_csv(
                path,
                dtype=str,
                keep_default_na=False,
                encoding="utf-8-sig",
                **kwargs,
            )
            if len(df_try.columns) >= 2:
                return df_try
            last_error = ValueError(f"Only {len(df_try.columns)} column(s) detected with {kwargs}")
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"Could not read coordinate CSV {path}. Last error: {last_error}")


def compute_map_extent_from_points(points):
    """
    Compute a tighter map extent from the sampling points.

    The goal is to zoom in relative to the original broad Arctic map while
    keeping some coastline/land visible. Since the sampling area is mostly
    Arctic seas north of the Russian coast, the latitude margins are asymmetric:
    a smaller southern margin keeps only a strip of land/coastline, and a larger
    northern margin keeps more open sea.
    """
    lon_min = float(points["Longitude"].min())
    lon_max = float(points["Longitude"].max())
    lat_min = float(points["Latitude"].min())
    lat_max = float(points["Latitude"].max())

    west = max(AUTO_EXTENT_MIN_WEST_LON, lon_min - POINT_MARGIN_LON_DEG)
    east = min(AUTO_EXTENT_MAX_EAST_LON, lon_max + POINT_MARGIN_LON_DEG)
    south = max(AUTO_EXTENT_MIN_SOUTH_LAT, lat_min - POINT_MARGIN_SOUTH_LAT_DEG)
    north = min(AUTO_EXTENT_MAX_NORTH_LAT, lat_max + POINT_MARGIN_NORTH_LAT_DEG)

    # A small safety guard: Cartopy behaves better when the extent is not too
    # narrow in either direction. This should not affect the present dataset.
    if east - west < 10:
        mid = 0.5 * (west + east)
        west = max(AUTO_EXTENT_MIN_WEST_LON, mid - 5)
        east = min(AUTO_EXTENT_MAX_EAST_LON, mid + 5)
    if north - south < 8:
        mid = 0.5 * (south + north)
        south = max(AUTO_EXTENT_MIN_SOUTH_LAT, mid - 4)
        north = min(AUTO_EXTENT_MAX_NORTH_LAT, mid + 4)

    return [round(west, 2), round(east, 2), round(south, 2), round(north, 2)]


# =============================================================================
# 3. NATURAL EARTH VECTOR BASEMAP HELPERS
# =============================================================================


def cartopy_data_search_dirs():
    """Return Cartopy data directories that may contain Natural Earth files."""
    dirs = []
    for key in ["pre_existing_data_dir", "data_dir"]:
        value = cartopy.config.get(key, None)
        if value:
            dirs.append(Path(value).expanduser())
    # Remove duplicates while preserving order.
    unique = []
    for d in dirs:
        if d not in unique:
            unique.append(d)
    return unique


def expected_natural_earth_path(resolution, category, name, base_dir):
    """Expected .shp path in a Cartopy Natural Earth cache directory."""
    return (
        Path(base_dir)
        / "shapefiles"
        / "natural_earth"
        / category
        / f"ne_{resolution}_{name}.shp"
    )


def is_natural_earth_cached(resolution, category, name):
    """Check whether a Natural Earth shapefile appears to be available locally."""
    for base_dir in cartopy_data_search_dirs():
        if expected_natural_earth_path(resolution, category, name, base_dir).exists():
            return True
    return False


def get_natural_earth_path(resolution, category, name, required=False):
    """
    Get the path to a Natural Earth shapefile.

    If the file is missing and ALLOW_NATURAL_EARTH_DOWNLOAD=True, Cartopy will
    try to download it. If the download fails, required layers raise a clear
    error; optional layers return None and are skipped.
    """
    if not ALLOW_NATURAL_EARTH_DOWNLOAD and not is_natural_earth_cached(resolution, category, name):
        msg = (
            f"Natural Earth layer is not cached locally: {resolution}/{category}/{name}. "
            "Install cartopy_offlinedata or set ALLOW_NATURAL_EARTH_DOWNLOAD=True."
        )
        if required:
            raise RuntimeError(msg)
        warnings.warn(msg + " Layer skipped.")
        return None

    try:
        return shapereader.natural_earth(
            resolution=resolution,
            category=category,
            name=name,
        )
    except Exception as exc:
        msg = (
            f"Could not access Natural Earth layer: {resolution}/{category}/{name}.\n"
            f"Original error: {type(exc).__name__}: {exc}\n\n"
            "Most likely reason: Cartopy tried to download Natural Earth data, "
            "but this Jupyter environment has no internet/DNS access.\n\n"
            "Recommended fix in terminal:\n"
            "    conda activate map_cartopy\n"
            "    mamba install -c conda-forge cartopy_offlinedata -y\n"
            "or:\n"
            "    conda install -c conda-forge cartopy_offlinedata -y\n"
        )
        if required:
            raise RuntimeError(msg) from exc
        warnings.warn(msg + "\nOptional layer skipped.")
        return None


def read_ne_geometries(resolution, category, name, required=False):
    """Read Natural Earth geometries now, so drawing/saving will not download later."""
    path = get_natural_earth_path(resolution, category, name, required=required)
    if path is None:
        return None
    try:
        reader = shapereader.Reader(path)
        return list(reader.geometries())
    except Exception as exc:
        msg = f"Could not read Natural Earth shapefile {path}: {type(exc).__name__}: {exc}"
        if required:
            raise RuntimeError(msg) from exc
        warnings.warn(msg + " Optional layer skipped.")
        return None


def load_basemap_geometries():
    """Load all vector basemap geometries before drawing the figures."""
    print("\nNatural Earth / Cartopy data directories:")
    for d in cartopy_data_search_dirs():
        print(f"  {d}")

    print(f"\nUsing Natural Earth resolution: {NATURAL_EARTH_RESOLUTION}")

    # Land is required because the reviewer explicitly requested a scientific map.
    land = read_ne_geometries(
        NATURAL_EARTH_RESOLUTION,
        "physical",
        "land",
        required=True,
    )

    # Optional layers improve appearance but should not crash the script.
    coastline = None
    lakes = None
    borders = None
    rivers = None

    if DRAW_COASTLINE:
        coastline = read_ne_geometries(
            NATURAL_EARTH_RESOLUTION,
            "physical",
            "coastline",
            required=False,
        )

    if DRAW_LAKES:
        lakes = read_ne_geometries(
            NATURAL_EARTH_RESOLUTION,
            "physical",
            "lakes",
            required=False,
        )

    if DRAW_COUNTRY_BORDERS:
        borders = read_ne_geometries(
            NATURAL_EARTH_RESOLUTION,
            "cultural",
            "admin_0_boundary_lines_land",
            required=False,
        )

    if DRAW_RIVERS:
        rivers = read_ne_geometries(
            NATURAL_EARTH_RESOLUTION,
            "physical",
            "rivers_lake_centerlines",
            required=False,
        )

    print("\nLoaded basemap layers:")
    print(f"  land:      {0 if land is None else len(land)} geometries")
    print(f"  coastline: {0 if coastline is None else len(coastline)} geometries")
    print(f"  lakes:     {0 if lakes is None else len(lakes)} geometries")
    print(f"  borders:   {0 if borders is None else len(borders)} geometries")
    print(f"  rivers:    {0 if rivers is None else len(rivers)} geometries")

    return {
        "land": land,
        "coastline": coastline,
        "lakes": lakes,
        "borders": borders,
        "rivers": rivers,
    }


def add_vector_basemap(ax, basemap_geometries):
    """Add crisp Natural Earth vector basemap layers."""
    ax.set_facecolor(OCEAN_COLOR)

    land = basemap_geometries.get("land")
    if land:
        land_feature = ShapelyFeature(
            land,
            DATA_PROJECTION,
            facecolor=LAND_COLOR,
            edgecolor=LAND_EDGE_COLOR,
            linewidth=0.35,
        )
        ax.add_feature(land_feature, zorder=LAND_ZORDER)

    lakes = basemap_geometries.get("lakes")
    if lakes:
        lake_feature = ShapelyFeature(
            lakes,
            DATA_PROJECTION,
            facecolor=LAKE_COLOR,
            edgecolor="0.55",
            linewidth=0.25,
        )
        ax.add_feature(lake_feature, zorder=LAKES_ZORDER)

    borders = basemap_geometries.get("borders")
    if borders:
        border_feature = ShapelyFeature(
            borders,
            DATA_PROJECTION,
            facecolor="none",
            edgecolor=BORDER_COLOR,
            linewidth=0.28,
            linestyle="-",
        )
        ax.add_feature(border_feature, zorder=BORDERS_ZORDER)

    rivers = basemap_geometries.get("rivers")
    if rivers:
        river_feature = ShapelyFeature(
            rivers,
            DATA_PROJECTION,
            facecolor="none",
            edgecolor="#6f9fc8",
            linewidth=0.25,
        )
        ax.add_feature(river_feature, zorder=RIVERS_ZORDER)

    coastline = basemap_geometries.get("coastline")
    if coastline:
        coastline_feature = ShapelyFeature(
            coastline,
            DATA_PROJECTION,
            facecolor="none",
            edgecolor="0.18",
            linewidth=0.45,
        )
        ax.add_feature(coastline_feature, zorder=COASTLINE_ZORDER)


# =============================================================================
# 4. MAP ANNOTATION HELPERS
# =============================================================================


def format_lon(lon):
    lon = float(lon)
    if lon < 0:
        return f"{abs(lon):g}°W"
    if lon > 0:
        return f"{lon:g}°E"
    return "0°"


def format_lat(lat):
    lat = float(lat)
    if lat < 0:
        return f"{abs(lat):g}°S"
    if lat > 0:
        return f"{lat:g}°N"
    return "0°"


def add_north_arrow(ax, x=0.935, y=0.155, size=0.085):
    """Add a north arrow in axes coordinates."""
    ax.annotate(
        "N",
        xy=(x, y + size),
        xytext=(x, y),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", lw=2.0, color="black", shrinkA=0, shrinkB=0),
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=ANNOTATION_ZORDER,
    )


def geodesic_end_point(lon0, lat0, azimuth_deg, distance_km):
    """Return lon/lat end point from a WGS84 or spherical-Earth geodesic."""
    if Geod is not None:
        geod = Geod(ellps="WGS84")
        lon1, lat1, _ = geod.fwd(lon0, lat0, azimuth_deg, distance_km * 1000.0)
        return lon1, lat1

    # Fallback: spherical approximation.
    lon1 = lon0 + distance_km / (111.32 * np.cos(np.deg2rad(lat0)))
    lat1 = lat0
    return lon1, lat1


def add_scalebar_geodesic(
    ax,
    extent,
    length_km=500,
    lon0=None,
    lat0=None,
    linewidth=4,
    text_offset_deg=1.2,
):
    """Add an approximate geodesic scale bar."""
    west, east, south, north = extent
    if lon0 is None:
        lon0 = west + 8
    if lat0 is None:
        lat0 = max(south + 4, min(62, north - 4))

    lon1, lat1 = geodesic_end_point(lon0, lat0, 90, length_km)

    ax.plot(
        [lon0, lon1],
        [lat0, lat1],
        transform=DATA_PROJECTION,
        color="black",
        lw=linewidth,
        solid_capstyle="butt",
        zorder=ANNOTATION_ZORDER,
    )

    tick_h = 0.7
    for lon in [lon0, lon1]:
        ax.plot(
            [lon, lon],
            [lat0 - tick_h / 2, lat0 + tick_h / 2],
            transform=DATA_PROJECTION,
            color="black",
            lw=linewidth * 0.75,
            zorder=ANNOTATION_ZORDER,
        )

    ax.text(
        (lon0 + lon1) / 2,
        lat0 + text_offset_deg,
        f"{length_km} km",
        transform=DATA_PROJECTION,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white")],
        zorder=ANNOTATION_ZORDER,
    )


def axes_fraction_to_lonlat(ax, x_axes, y_axes):
    """
    Convert an axes-fraction coordinate to lon/lat degrees.

    This helper lets us place the scale bar visually in a fixed map corner and
    still calculate its real geodesic length from the map projection.
    """
    display_xy = ax.transAxes.transform((x_axes, y_axes))
    proj_xy = ax.transData.inverted().transform(display_xy)
    lon, lat = DATA_PROJECTION.transform_point(proj_xy[0], proj_xy[1], ax.projection)
    return float(lon), float(lat)


def geodesic_distance_km(lon0, lat0, lon1, lat1):
    """Return geodesic distance in km between two lon/lat points."""
    if Geod is not None:
        geod = Geod(ellps="WGS84")
        _, _, distance_m = geod.inv(lon0, lat0, lon1, lat1)
        return abs(distance_m) / 1000.0

    # Fallback: haversine distance.
    radius_km = 6371.0
    lon0r, lat0r, lon1r, lat1r = map(np.deg2rad, [lon0, lat0, lon1, lat1])
    dlon = lon1r - lon0r
    dlat = lat1r - lat0r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat0r) * np.cos(lat1r) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return radius_km * c


def distance_between_axes_points_km(ax, x0, y0, x1, y1):
    """Geodesic distance between two screen/axes-fraction points on the map."""
    lon0, lat0 = axes_fraction_to_lonlat(ax, x0, y0)
    lon1, lat1 = axes_fraction_to_lonlat(ax, x1, y1)
    if not np.all(np.isfinite([lon0, lat0, lon1, lat1])):
        return np.nan
    return geodesic_distance_km(lon0, lat0, lon1, lat1)


def find_axes_scalebar_endpoints(ax, length_km, corner="lower right"):
    """
    Find axes-fraction endpoints for a horizontal scale bar in a chosen corner.

    The line is horizontal on the page/screen. Its width is found by binary
    search so that the geodesic distance between the two endpoints is close to
    length_km at this part of the map.
    """
    corner = corner.lower().strip()
    if corner not in {"lower right", "lower left", "upper right", "upper left"}:
        raise ValueError(
            "SCALEBAR_CORNER must be one of: 'lower right', 'lower left', "
            "'upper right', 'upper left'."
        )

    lower = corner.startswith("lower")
    right = corner.endswith("right")

    y = SCALEBAR_MARGIN_Y_AXES if lower else 1.0 - SCALEBAR_MARGIN_Y_AXES
    x_margin = SCALEBAR_MARGIN_X_AXES

    # Keep the scale bar well inside the map frame.
    x_min_allowed = 0.04
    x_max_allowed = 0.96

    if right:
        fixed_x = 1.0 - x_margin
        max_width = fixed_x - x_min_allowed
    else:
        fixed_x = x_margin
        max_width = x_max_allowed - fixed_x

    # Check whether the requested scale fits. If not, use the maximum possible
    # width and label the actual distance.
    if right:
        max_distance = distance_between_axes_points_km(ax, fixed_x - max_width, y, fixed_x, y)
    else:
        max_distance = distance_between_axes_points_km(ax, fixed_x, y, fixed_x + max_width, y)

    if not np.isfinite(max_distance) or max_distance <= 0:
        raise RuntimeError(
            "Could not compute scale-bar length from the projection. "
            "Try another SCALEBAR_CORNER or increase MAP_EXTENT."
        )

    target_distance = min(float(length_km), max_distance)

    lo, hi = 0.001, max_width
    for _ in range(50):
        width = (lo + hi) / 2.0
        if right:
            x0, x1 = fixed_x - width, fixed_x
        else:
            x0, x1 = fixed_x, fixed_x + width

        dist = distance_between_axes_points_km(ax, x0, y, x1, y)
        if not np.isfinite(dist):
            hi = width
            continue
        if dist < target_distance:
            lo = width
        else:
            hi = width

    width = (lo + hi) / 2.0
    if right:
        x0, x1 = fixed_x - width, fixed_x
    else:
        x0, x1 = fixed_x, fixed_x + width

    actual_distance = distance_between_axes_points_km(ax, x0, y, x1, y)
    return x0, x1, y, actual_distance


def add_scalebar_in_corner(ax, length_km=500, corner="lower right"):
    """
    Add a scale bar in the selected map corner.

    Unlike the previous lon/lat-positioned scale bar, this one is anchored in an
    axes corner, so it stays visually in the corner on both polar map variants.
    The endpoint separation is adjusted to represent the requested geodesic
    distance at that local map position.
    """
    x0, x1, y, actual_distance = find_axes_scalebar_endpoints(
        ax=ax,
        length_km=length_km,
        corner=corner,
    )

    lower = corner.lower().strip().startswith("lower")
    tick_h = SCALEBAR_TICK_HEIGHT_AXES
    text_dy = 0.018 if lower else -0.022
    text_va = "bottom" if lower else "top"

    # White underlay improves readability over land/sea/gridlines.
    ax.plot(
        [x0, x1],
        [y, y],
        transform=ax.transAxes,
        color="white",
        lw=SCALEBAR_LINEWIDTH + 3.0,
        solid_capstyle="butt",
        zorder=ANNOTATION_ZORDER - 1,
        clip_on=False,
    )
    ax.plot(
        [x0, x1],
        [y, y],
        transform=ax.transAxes,
        color="black",
        lw=SCALEBAR_LINEWIDTH,
        solid_capstyle="butt",
        zorder=ANNOTATION_ZORDER,
        clip_on=False,
    )

    for x in [x0, x1]:
        ax.plot(
            [x, x],
            [y - tick_h / 2, y + tick_h / 2],
            transform=ax.transAxes,
            color="white",
            lw=SCALEBAR_LINEWIDTH,
            zorder=ANNOTATION_ZORDER - 1,
            clip_on=False,
        )
        ax.plot(
            [x, x],
            [y - tick_h / 2, y + tick_h / 2],
            transform=ax.transAxes,
            color="black",
            lw=max(1.0, SCALEBAR_LINEWIDTH * 0.65),
            zorder=ANNOTATION_ZORDER,
            clip_on=False,
        )

    # Normally this will be exactly length_km. If the map corner is too narrow,
    # the label will reflect the actual rounded distance.
    if np.isfinite(actual_distance) and abs(actual_distance - length_km) / length_km < 0.05:
        label_distance = int(round(length_km))
    else:
        label_distance = int(round(actual_distance / 50.0) * 50)

    ax.text(
        (x0 + x1) / 2.0,
        y + text_dy,
        f"{label_distance} km",
        transform=ax.transAxes,
        ha="center",
        va=text_va,
        fontsize=12,
        fontweight="bold",
        color="black",
        path_effects=[pe.withStroke(linewidth=3.5, foreground="white")],
        zorder=ANNOTATION_ZORDER,
        clip_on=False,
    )


def add_label(ax, text, lon, lat, fontsize=14, weight="normal", ha="center"):
    """Add a map label with a white outline for readability."""
    ax.text(
        lon,
        lat,
        text,
        transform=DATA_PROJECTION,
        fontsize=fontsize,
        fontweight=weight,
        ha=ha,
        va="center",
        color="black",
        path_effects=[pe.withStroke(linewidth=3.5, foreground="white")],
        zorder=MAP_LABEL_ZORDER,
    )


def add_manual_graticule_labels_for_polar(ax, extent):
    """Add coordinate labels manually for the polar projection."""
    west, east, south, north = extent

    label_lat = south + 1.5
    for lon in MERIDIANS:
        if west <= lon <= east:
            ax.text(
                lon,
                label_lat,
                format_lon(lon),
                transform=DATA_PROJECTION,
                ha="center",
                va="bottom",
                fontsize=9.5,
                color="0.20",
                path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
                zorder=GRATICULE_LABEL_ZORDER,
                clip_on=True,
            )

    label_lon = west + 2.0
    for lat in PARALLELS:
        if south <= lat <= north:
            ax.text(
                label_lon,
                lat,
                format_lat(lat),
                transform=DATA_PROJECTION,
                ha="left",
                va="center",
                fontsize=9.5,
                color="0.20",
                path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
                zorder=GRATICULE_LABEL_ZORDER,
                clip_on=True,
            )


def add_ticks_for_straight_map(ax, extent):
    """Add straight longitude/latitude axis ticks for the Plate Carree map."""
    west, east, south, north = extent
    x_ticks = [x for x in MERIDIANS if west <= x <= east]
    y_ticks = [y for y in PARALLELS if south <= y <= north]

    ax.set_xticks(x_ticks, crs=DATA_PROJECTION)
    ax.set_yticks(y_ticks, crs=DATA_PROJECTION)
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="°"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="°"))
    ax.tick_params(axis="both", labelsize=10, direction="out", length=4, width=0.8)
    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)


def add_graticule(ax):
    """Add gridlines without Cartopy automatic labels."""
    gl = ax.gridlines(
        crs=DATA_PROJECTION,
        draw_labels=False,
        linewidth=0.45,
        color=GRID_COLOR,
        alpha=0.65,
        linestyle="--",
        zorder=GRID_ZORDER,
        x_inline=False,
        y_inline=False,
    )
    gl.xlocator = mticker.FixedLocator(MERIDIANS)
    gl.ylocator = mticker.FixedLocator(PARALLELS)

    # Explicitly ensure that all Cartopy Gridliner labels are off.
    # This avoids the Shapely/GEOS gridliner error seen in the previous log.
    for attr in ["top_labels", "bottom_labels", "left_labels", "right_labels", "geo_labels"]:
        if hasattr(gl, attr):
            setattr(gl, attr, False)


def add_standard_map_annotations(ax, extent):
    """Add sea labels, north arrow, scale bar and basemap source text."""
    add_label(ax, "Arctic Ocean", 88, 82.6, fontsize=21, weight="bold")
    add_label(ax, "Svalbard", 17, 78.0, fontsize=12)
    add_label(ax, "Barents Sea", 43, 73.6, fontsize=14)
    add_label(ax, "Kara Sea", 76, 74.7, fontsize=14)
    add_label(ax, "Laptev Sea", 116, 76.2, fontsize=14)
    add_label(ax, "East Siberian\nSea", 150, 74.5, fontsize=13)

    # Move the north arrow away from the lower-right scale-bar corner.
    add_north_arrow(ax, x=0.935, y=0.78, size=0.085)

    # Corner-anchored scale bar. Change SCALEBAR_CORNER above if you prefer
    # another corner.
    add_scalebar_in_corner(ax, length_km=SCALEBAR_LENGTH_KM, corner=SCALEBAR_CORNER)

    ax.text(
        0.012,
        0.012,
        BASEMAP_SOURCE_TEXT,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.8,
        color="0.15",
        bbox=dict(facecolor="white", edgecolor="0.7", boxstyle="round,pad=0.25", alpha=0.85),
        zorder=ANNOTATION_ZORDER,
    )


# =============================================================================
# 5. SAVING AND MAP DRAWING
# =============================================================================


def save_figure(fig, output_basename):
    """
    Save the figure and print the saved paths.

    This is how the picture is saved:
        fig.savefig("file_name.png", dpi=600)
    The script does this automatically for PNG, PDF and SVG.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved_files = []

    if SAVE_PNG:
        path = OUTPUT_DIR / f"{output_basename}.png"
        fig.savefig(path, dpi=OUTPUT_DPI)
        saved_files.append(path)

    if SAVE_PDF:
        path = OUTPUT_DIR / f"{output_basename}.pdf"
        fig.savefig(path)
        saved_files.append(path)

    if SAVE_SVG:
        path = OUTPUT_DIR / f"{output_basename}.svg"
        fig.savefig(path)
        saved_files.append(path)

    print(f"\nSaved files for {output_basename}:")
    for path in saved_files:
        print(f"  {path.resolve()}")

    return saved_files


def make_map(points_to_plot, basemap_geometries, projection, output_basename, map_kind):
    """Draw and save one map variant."""
    if map_kind not in {"curved", "topdown"}:
        raise ValueError("map_kind must be 'curved' or 'topdown'.")

    # Both variants are true polar/top-down maps. The second variant replaces
    # the old rectangular PlateCarree map that visually looked tilted relative
    # to the pole.
    if map_kind == "topdown":
        fig = plt.figure(figsize=(12.5, 10.0))
    else:
        fig = plt.figure(figsize=(13.5, 8.5))

    ax = plt.axes(projection=projection)
    ax.set_extent(MAP_EXTENT, crs=DATA_PROJECTION)

    # Crisp vector basemap, no blurred raster background.
    add_vector_basemap(ax, basemap_geometries)
    add_graticule(ax)

    # Automatic Cartopy gridliner labels are disabled because they previously
    # triggered Shapely/GEOS errors. Manual labels are stable for both polar maps.
    add_manual_graticule_labels_for_polar(ax, MAP_EXTENT)

    # Sampling points.
    ax.scatter(
        points_to_plot["Longitude"],
        points_to_plot["Latitude"],
        s=POINT_SIZE,
        transform=DATA_PROJECTION,
        facecolor=POINT_FACE_COLOR,
        edgecolor=POINT_EDGE_COLOR,
        linewidth=0.8,
        alpha=0.95,
        zorder=POINT_ZORDER,
    )

    add_standard_map_annotations(ax, MAP_EXTENT)

    try:
        ax.spines["geo"].set_linewidth(1.0)
    except Exception:
        pass

    # Do NOT use plt.tight_layout() or bbox_inches="tight" here.
    # In some Cartopy/Shapely versions this can trigger geometry/gridliner errors.
    if map_kind == "topdown":
        fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.98)
    else:
        fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.98)

    save_figure(fig, output_basename)
    return fig, ax


# =============================================================================
# 6. READ AND CLEAN COORDINATES
# =============================================================================

csv_file = find_coordinate_file(CSV_FILE_CANDIDATES)
print(f"Reading coordinates from: {csv_file}")

df = read_coordinate_csv(csv_file)
lat_col, lon_col = infer_coordinate_columns(df)

points = df.copy()
points["Latitude"] = parse_decimal_series(points[lat_col])
points["Longitude"] = parse_decimal_series(points[lon_col])
points = points.dropna(subset=["Latitude", "Longitude"]).copy()

# Basic longitude normalization, useful if values are provided as 0..360.
points.loc[points["Longitude"] > 180, "Longitude"] -= 360

if AUTO_COMPUTE_MAP_EXTENT_FROM_POINTS:
    MAP_EXTENT = compute_map_extent_from_points(points)
    print(f"Auto-computed map extent: {MAP_EXTENT}")
else:
    print(f"Manual map extent: {MAP_EXTENT}")

if FILTER_POINTS_TO_EXTENT:
    west, east, south, north = MAP_EXTENT
    points_to_plot = points[
        points["Longitude"].between(west, east)
        & points["Latitude"].between(south, north)
    ].copy()
else:
    points_to_plot = points.copy()

print(f"Total valid points in CSV: {len(points)}")
print(f"Points plotted: {len(points_to_plot)}")
print(
    "Coordinate range in CSV: "
    f"lat {points['Latitude'].min():.2f}..{points['Latitude'].max():.2f}, "
    f"lon {points['Longitude'].min():.2f}..{points['Longitude'].max():.2f}"
)

if len(points_to_plot) == 0:
    raise ValueError("No points to plot. Check MAP_EXTENT and FILTER_POINTS_TO_EXTENT.")


# =============================================================================
# 7. LOAD BASEMAP DATA, DRAW AND SAVE TWO MAP VARIANTS
# =============================================================================

basemap_geometries = load_basemap_geometries()

topdown_projection = ccrs.AzimuthalEquidistant(
    central_longitude=POLAR_CENTRAL_LONGITUDE,
    central_latitude=90,
)
fig_topdown, ax_topdown = make_map(
    points_to_plot=points_to_plot,
    basemap_geometries=basemap_geometries,
    projection=topdown_projection,
    output_basename="arctic_sampling_map_topdown",
    map_kind="topdown",
)

print("\nDone.")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print("Recommended journal file: PNG at 600 dpi.")
print("Editable files: PDF and SVG.")

if SHOW_FIGURES_IN_NOTEBOOK:
    plt.show()
else:
    plt.close(fig_topdown)